# 7000LBSBAI - Programming for Business Analytics
## Week 10 - Code-along: Regression Models for EDA
### 📋 ANSWER VERSION: for checking code and independent study

---

**Module:** Programming for Business Analytics  
**Week:** 10  
**Time:** About 60 minutes (four blocks, with your tutor)  
**Topics covered:** Multiple linear regression · Logistic regression and odds ratios · Choosing predictors: overlap and leakage · Gemini as a reviewer

---

### How to use this notebook

| Symbol | Meaning |
|--------|---------|
| 👀 Watch | Your tutor explains this cell first -> then you run it |
| 🆘 Solution | The answer to an Adapt task, if you get stuck |
| ✍️ Worked example | Code is already written -> run it and study the output |
| ➡️ Your turn | Write or complete the code in this cell |
| 💬 What does it mean? | Write a short interpretation of your results |
| 🤖 AI activity | A guided activity using Gemini in Colab |
| 🔴 Extension | For students who finish early (optional) |
| 📝 Hint/Tip | Some friendly help and advice |

To run a cell: click on it and press **Shift + Enter**.

---

## ⚠️ BEFORE YOU CONTINUE

### Save a copy of this notebook

Make sure your notebook is saved to your Google Drive:
- **File → Save a copy in Drive**
- Name it: `Week10_CodeAlong_Regression_Models_YourName.ipynb`

---

### Turn off AI-powered in-line completions

Go to **Tools** -> **Settings** -> **AI Assistance**

> Uncheck `☐ Show AI-powered inline completions`  

> Check `☑️ Hide generative AI features`

---
## How this code-along works

The same four-block format: **👀 Watch → Run → ➡️ Adapt → 💬 Interpret**.

- Each block has a **time limit**. When the time is up, we move on together.
- Stuck on an Adapt task? Open the **🆘 solution**, run it, and go straight to the interpretation.
- Finished early? Try the **🔴 Go further** challenge.

---
## Which regression model?

Last week you fitted a model with **one** predictor. This week you learn the two models you are most likely to need in an EDA, and how to choose between them:

| Your outcome is… | Example | Use | statsmodels |
|---|---|---|---|
| **A number** | delivery time, spend | **Linear** regression | `smf.ols()` |
| **Yes or no** (1/0) | late or on time, cancelled, churned | **Logistic** regression | `smf.logit()` |

You know the theory from your statistics module. Here you learn to **fit** each model in Python, **read** the output, and **explain** it in plain English.

| Block | What you learn |
|---|---|
| 1 | Linear regression with **several** predictors |
| 2 | **Logistic** regression for a yes/no outcome |
| 3 | **Choosing predictors** safely: overlap and leakage |
| 4 | Using **Gemini as a reviewer** to improve your code |

We will use the **SwiftDeliver** orders dataset throughout.

---
## Set up: run this cell

This cell loads and cleans SwiftDeliver with the Week 8 function, and creates the `late` column.

In [ ]:
# ✍️ Set up: just run this cell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

# Set Seaborn chart style
sns.set_theme(style='whitegrid')

# Function to clean SwiftDeliver orders data
def clean_orders(data):
    """Clean SwiftDeliver orders: the Week 6 cleaning steps in one place."""
    data = data.copy()
    data = data.drop_duplicates()
    data['restaurant_type'] = data['restaurant_type'].str.strip().str.title()
    data = data[data['delivery_time_mins'] > 0]
    return data

# Import SwiftDeliver orders
raw = pd.read_csv("https://raw.githubusercontent.com/7000lbsbai/7000LBSBAI/main/datasets/swiftdeliver_orders.csv")

# Use clean_orders function to clean raw dataset
df = clean_orders(raw)

# Create 'late' column: 1 = delivery > 30 mins; 0 = delivery <= 30mins
df['late'] = np.where(df['delivery_time_mins'] > 30, 1, 0)

# Check all processed correctly
print("Data loaded!")
print(f"SwiftDeliver: {df.shape[0]} orders")

---
# Block 1: Linear regression with several predictors
*about 15 minutes*

In Week 9, distance alone explained 74% of the variation in delivery times. But rain and driver experience matter too. A **multiple** regression includes several predictors at once, separated by `+`:

```python
smf.ols('delivery_time_mins ~ distance_km + C(weather) + driver_experience_yrs', data=df).fit()
```

- **Each coefficient is the effect of that predictor, holding the others constant.** For example, the effect of rain *for deliveries of the same distance, by equally experienced drivers*.
- **`C()` marks a category.** statsmodels compares each group with a **reference group** (the first alphabetically: here **Dry**). The coefficient for `C(weather)[T.Rain]` is the difference between rain and dry.
- **Adjusted R²** is R² with a small penalty for each extra predictor. Use it to **compare** models with different numbers of predictors: it only rises if a new predictor genuinely helps.

In [ ]:
# 👀 Block 1: delivery time explained by distance, weather and driver experience
# Watch the demo first, then run this cell yourself

# 1. Fit the model with three predictors
model = smf.ols('delivery_time_mins ~ distance_km + C(weather) + driver_experience_yrs', data=df).fit()

# 2. The key numbers: coefficients, p-values and 95% confidence intervals
# Using pd.DataFrame puts our results into a table
results = pd.DataFrame({
    'coefficient': model.params,
    'p_value': model.pvalues,
    'lower_95': model.conf_int()[0],
    'upper_95': model.conf_int()[1]
}).round(3)                         # Round outputs to 3 decimal places
print(results)
print()

# 3. Compare with the Week 9 model, which used distance only
simple = smf.ols('delivery_time_mins ~ distance_km', data=df).fit()
print(f"Adjusted R², distance only:    {simple.rsquared_adj:.3f}")
print(f"Adjusted R², three predictors: {model.rsquared_adj:.3f}")

> 📝 You can still use `print(model.summary())` to see everything. The table above picks out the parts you need, and it is a good format to copy into a report.

### ➡️ Adapt: add restaurant type

Fit a fourth model that also includes `C(restaurant_type)`. Print its coefficients and p-values (rounded to 3 decimal places) and its adjusted R². Does restaurant type improve the model?

> 📝 **Hint:** add `+ C(restaurant_type)` to the formula, and use a new name, e.g. `model4`, so you keep `model`.

In [ ]:
# ➡️ Adapt: add restaurant type
model4 = smf.ols('delivery_time_mins ~ distance_km + C(weather) + driver_experience_yrs + C(restaurant_type)', data=df).fit()

results = pd.DataFrame({
    'coefficient': model4.params,
    'p_value': model4.pvalues
}).round(3)
print(results)
print()
print(f"Adjusted R²: {model4.rsquared_adj:.3f}")

# Adjusted R² stays at 0.884, and every restaurant type has p > 0.05: no improvement

<details>
<summary><b>🆘 Stuck? Click here for the solution to "add restaurant type"</b></summary>

Copy this code into the cell above, run it, then move on to the interpretation.

```python
model4 = smf.ols('delivery_time_mins ~ distance_km + C(weather) + driver_experience_yrs + C(restaurant_type)', data=df).fit()
results = pd.DataFrame({
    'coefficient': model4.params,
    'p_value': model4.pvalues
}).round(3)
print(results)
print()
print(f"Adjusted R²: {model4.rsquared_adj:.3f}")
```
</details>

### 💬 What does it mean?

- Explain each coefficient in the three-predictor model in plain English.
- How much did adding weather and driver experience improve the model?
- Should SwiftDeliver keep restaurant type in the model? Why or why not?

**✏️ MODEL ANSWER:**

- Holding the other factors constant:
  - Each extra **kilometre** adds about **3.4 minutes**,
  - **Rain** adds about **7 minutes** compared with a dry day,
  - Each extra **year of driver experience** saves about **0.5 minutes**.
- All three have p-values well below 0.05, and none of the confidence intervals include zero.
- Adding weather and experience raised the adjusted R² from **0.743 to 0.884**: the model now explains about 88% of the variation in delivery times.
- Restaurant type does **not** help: the adjusted R² stays at 0.884 and every restaurant type has p > 0.05, so it can be left out. A simpler model that explains just as much is better.

### 🔴 Go further: check the model with a residual plot

A **residual** is the difference between an actual value and the model's prediction.  
If a linear model fits well, the residuals are scattered evenly around zero with no pattern.  
Create a scatter plot of `model.fittedvalues` against `model.resid`, with a red dashed, horizontal line (`plt.axhline()`) at 0.  
Don't forget to give your chart a title and axis labels!

Is the linear model a reasonable fit?

In [ ]:
# 🔴 Go further: check the model with a residual plot
plt.figure(figsize=(10, 5))
plt.scatter(model.fittedvalues, model.resid, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.title('Residuals against predicted delivery time')
plt.xlabel('Predicted delivery time (minutes)')
plt.ylabel('Residual (actual minus predicted)')
plt.show()

# An even band around zero, with no curve or funnel shape: the linear model is a reasonable fit

---
# Block 2: Logistic regression for a yes/no outcome
*about 15 minutes*

When the outcome is **yes or no** (1/0), such as whether a delivery was late, linear regression is not suitable: it can predict values below 0 or above 1.  
**Logistic regression** predicts the **probability** of a 1, which always stays between 0 and 1.

```python
smf.logit('late ~ distance_km + C(weather)', data=df).fit()
```

The raw coefficients are on a scale that is hard to explain (log-odds). To make them readable, turn them into **odds ratios** with `np.exp()`:

| Odds ratio | Meaning |
|---|---|
| **Above 1** | Increases the odds of the outcome (e.g. 2 = doubles the odds) |
| **1** | No effect |
| **Below 1** | Decreases the odds (e.g. 0.8 = 20% lower odds) |

> 📝 The outcome must be **1/0**, not "Yes"/"No".  
> `.fit(disp=0)` stops statsmodels printing its progress messages.

In [ ]:
# 👀 Block 2: the chance of a late delivery, from distance and weather
# Watch the demo first, then run this cell yourself

# 1. Fit a logistic regression: late (1/0) explained by distance and weather
logit_model = smf.logit('late ~ distance_km + C(weather)', data=df).fit(disp=0)

# 2. Odds ratios (with 95% confidence intervals) and p-values
odds = pd.DataFrame({
    'odds_ratio': np.exp(logit_model.params),
    'lower_95': np.exp(logit_model.conf_int()[0]),
    'upper_95': np.exp(logit_model.conf_int()[1]),
    'p_value': logit_model.pvalues
}).round(3)
print(odds)
print()

# 3. Predicted probability of a late delivery for some example orders
examples = pd.DataFrame({'distance_km': [2, 2, 4, 4, 6, 6],
                         'weather': ['Dry', 'Rain', 'Dry', 'Rain', 'Dry', 'Rain']})
examples['chance_late_%'] = (logit_model.predict(examples) * 100).round(1)
print(examples)

> 📝 Ignore the odds ratio for the **Intercept**: it is the starting point of the model, not an effect to interpret.

### ➡️ Adapt: add driver experience

Fit a logistic model of `late` with `distance_km`, `C(weather)` **and** `driver_experience_yrs`.  
Print the odds ratios and p-values, rounded to 3 decimal places.

> 📝 **Hint:** the same as steps 1 and 2 of the Watch cell, with `+ driver_experience_yrs` added.  
> Use a new name, e.g. `logit_model2`.

In [ ]:
# ➡️ Adapt: add driver experience
logit_model2 = smf.logit('late ~ distance_km + C(weather) + driver_experience_yrs', data=df).fit(disp=0)

odds = pd.DataFrame({
    'odds_ratio': np.exp(logit_model2.params),
    'p_value': logit_model2.pvalues
}).round(3)
print(odds)

# Experience: odds ratio 0.764, p < 0.05. Each extra year cuts the odds of being late by about 24%

<details>
<summary><b>🆘 Stuck? Click here for the solution to "add driver experience"</b></summary>

Copy this code into the cell above, run it, then move on to the interpretation.

```python
logit_model2 = smf.logit('late ~ distance_km + C(weather) + driver_experience_yrs', data=df).fit(disp=0)
odds = pd.DataFrame({
    'odds_ratio': np.exp(logit_model2.params),
    'p_value': logit_model2.pvalues
}).round(3)
print(odds)
```
</details>

### 💬 What does it mean?

- Explain the odds ratios for distance and rain in plain English.
- Look at the predicted probabilities. Which orders are almost certain to be late, and which are almost never late?
- What does the odds ratio for driver experience tell SwiftDeliver?

**✏️ MODEL ANSWER:**

- Each extra **kilometre** multiplies the odds of a late delivery by about **5**, and **rain** multiplies them by about **18**, holding the other factor constant.
- Odds ratios this large show both are very strong predictors (both p < 0.05).
- The predicted probabilities make this easier to explain:
  - A **2 km** delivery in dry weather has under **1%** chance of being late,
  - At **4 km in the rain** the chance is about **77%**,
  - A **6 km** delivery in the rain is almost certain to be late (**99%**).
- Driver experience has an odds ratio of about **0.76**: each extra year of experience **lowers** the odds of a late delivery by about **24%**.
- SwiftDeliver could give longer rainy-day deliveries to its most experienced drivers.
- Predicted probabilities are often the clearest way to explain a logistic model to a manager.

### 🔴 Go further: draw the probability curves

Create 100 distances from 0 to 10 km with `distances = np.linspace(0, 10, 100)`.  
Use a for loop to predict the chance of being late for each distance in **dry** and **rainy** weather, and draw both lines on one chart.  
Where does each line pass 50%?

In [ ]:
# 🔴 Go further: draw the probability curves
distances = np.linspace(0, 10, 100)

plt.figure(figsize=(10, 5))
for weather in ['Dry', 'Rain']:
    grid = pd.DataFrame({'distance_km': distances, 'weather': weather})
    plt.plot(distances, logit_model.predict(grid) * 100, linewidth=3, label=weather)
plt.axhline(50, color='grey', linestyle='--')
plt.title('Predicted chance of a late delivery')
plt.xlabel('Distance (km)')
plt.ylabel('Chance of being late (%)')
plt.legend()
plt.show()

# The S-shaped curves cross 50% at about 5.0 km in dry weather and about 3.2 km in the rain

---
# Block 3: Choosing predictors: overlap and leakage
*about 15 minutes*

More predictors are not always better. Two problems are common in real business data:

| Problem | What it is | What goes wrong |
|---|---|---|
| **Overlap** | Two predictors carry almost the same information (e.g. one is calculated from the other) | The model cannot tell them apart: coefficients become unstable or even flip sign |
| **Leakage** | A predictor is made from the outcome, or is only known after it | The model looks perfect, but it tells you nothing useful |

To demonstrate overlap, the cell below adds a **made-up** column, `distance_miles`: the same distances in miles, with tiny recording differences.

In [ ]:
# 👀 Block 3: what overlap and leakage do to a model
# Watch the demo first, then run this cell yourself

# 1. OVERLAP: a made-up column that is almost the same as distance_km
rng = np.random.default_rng(3)
df['distance_miles'] = (df['distance_km'] * 0.621 + rng.normal(0, 0.05, len(df))).round(2)
print(f"Correlation between km and miles: {df['distance_km'].corr(df['distance_miles']):.3f}")
print()

overlap_model = smf.ols('delivery_time_mins ~ distance_km + distance_miles', data=df).fit()
print(pd.DataFrame({'coefficient': overlap_model.params, 'p_value': overlap_model.pvalues}).round(3))
print(f"R²: {overlap_model.rsquared:.3f}   (distance_km on its own: 0.744)")
print()

# 2. LEAKAGE: late was created FROM delivery time, so delivery time 'predicts' it perfectly
print(pd.crosstab(df['delivery_time_mins'] > 30, df['late']))

> 📝 With both distance columns, distance in km suddenly looks **negative** and **not significant**, even though R² barely changes. The model is splitting one effect between two near-identical columns, and the result is meaningless.

### ➡️ Adapt: check your candidate predictors

Before fitting a model, check for overlap.  
Draw a correlation heatmap of these candidate predictors: `distance_km`, `distance_miles`, `driver_experience_yrs`, `order_value` and `tip`.  
Which pairs are strongly correlated (above about 0.8)?

> 📝 **Hint:** Week 6, the correlation heatmap. Use `df[candidates].corr()`.

In [ ]:
# ➡️ Adapt: check your candidate predictors
candidates = ['distance_km', 'distance_miles', 'driver_experience_yrs', 'order_value', 'tip']

plt.figure(figsize=(8, 6))
sns.heatmap(df[candidates].corr().round(2), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlations between candidate predictors')
plt.tight_layout()
plt.show()

# Only distance_km and distance_miles (1.00). Order value and tip are moderately related (0.54):
# fine to use together, but worth mentioning

<details>
<summary><b>🆘 Stuck? Click here for the solution to "check your candidate predictors"</b></summary>

Copy this code into the cell above, run it, then move on to the interpretation.

```python
candidates = ['distance_km', 'distance_miles', 'driver_experience_yrs', 'order_value', 'tip']
plt.figure(figsize=(8, 6))
sns.heatmap(df[candidates].corr().round(2), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlations between candidate predictors')
plt.tight_layout()
plt.show()
```
</details>

### 💬 What does it mean?

- What happened to the distance coefficients when both distance columns were included? Why?
- Why would a model predicting `late` from `delivery_time_mins` be useless, even though it would be perfect?
- What checks would you do before choosing the predictors for a model of a yes/no outcome?

**✏️ MODEL ANSWER:**

- With both columns, the distance in km became **negative (-0.55)** and **not significant (p = 0.85)**, even though R² barely changed.
- The two columns carry the **same information**, so the model cannot separate their effects, and splits them in an unstable, meaningless way. Keep only one of any pair that is very strongly correlated or calculated from another.
- A model predicting `late` from delivery time would be **perfect but useless**: `late` was *created* from delivery time, so the model would only rediscover our own definition. That is **leakage**.
- Before choosing predictors, I would check that none is **made from the outcome** or only known **after** it (such as a date the event happened), and check a **correlation heatmap** of the candidates for overlapping pairs.

### 🔴 Go further: drop the copy

Remove `distance_miles` from the data frame with `df = df.drop(columns='distance_miles')`, so it cannot be used by mistake later. Check it has gone.

In [ ]:
# 🔴 Go further: drop the copy
df = df.drop(columns='distance_miles')
print('distance_miles' in df.columns)

# False: a good habit is to remove any column you have decided not to use

---
# Block 4: 🤖 Gemini as a reviewer
*about 15 minutes*

In Week 8 Gemini **explained** code; in Week 9 it **wrote** code that you checked. This week it **reviews** code that already works, and suggests how to make it clearer.

Clear code matters in Assessment 2: the rubric rewards **readable code with comments** and a notebook that reads as a document.

The cell below **works**, but it is hard to read: short names, no comments, unrounded output.

### The activity

1. Run the cell, and look at the output.
2. Copy the code into the **Gemini** chat (✨), with this prompt:

> *"Review this Python code for readability. Suggest improvements to variable names, comments and how the output is presented. Do not change what it calculates. Explain each suggestion."*

3. Decide which suggestions to **accept** and which to **reject**, and why.
4. Write your improved version in the Adapt cell, and **check the numbers are the same**.

In [ ]:
# 👀 Block 4: working code that is hard to read
# Watch the demo first, then run this cell yourself

m = smf.logit('late ~ distance_km + C(weather)', data=df).fit(disp=0)
r = np.exp(m.params)
p = m.pvalues
for i in range(len(r)):
    print(r.index[i], r.iloc[i], p.iloc[i])

### ➡️ Adapt: your reviewed version

Write an improved version using the suggestions you accepted. It must produce **the same numbers**. Then add a `# Changes I made:` comment listing what you changed.

> 📝 **Hint:** start from the Watch cell in Block 2: it already shows a clear way to build an odds ratio table.

In [ ]:
# ➡️ Adapt: your reviewed version
# Fit a logistic regression: is a delivery late (1/0), given its distance and the weather?
late_model = smf.logit('late ~ distance_km + C(weather)', data=df).fit(disp=0)

# Turn the coefficients into odds ratios, and put them in a table with the p-values
odds_table = pd.DataFrame({
    'odds_ratio': np.exp(late_model.params),
    'p_value': late_model.pvalues
}).round(3)

# Remove the intercept, which is not an effect to interpret
odds_table = odds_table.drop(index='Intercept')
print(odds_table)

# Changes I made:
# - Clear names (late_model, odds_table) instead of m, r and p
# - Comments saying what each step does and why
# - A table instead of a loop, rounded to 3 decimal places
# - Removed the intercept row
# Rejected: Gemini suggested a function. Useful later, but more than this needs.
# Check: odds ratios 5.091 (distance) and 18.332 (rain), the same as the original

<details>
<summary><b>🆘 Stuck? Click here for the solution to "your reviewed version"</b></summary>

Copy this code into the cell above, run it, then move on to the interpretation.

```python
late_model = smf.logit('late ~ distance_km + C(weather)', data=df).fit(disp=0)
odds_table = pd.DataFrame({
    'odds_ratio': np.exp(late_model.params),
    'p_value': late_model.pvalues
}).round(3)
print(odds_table.drop(index='Intercept'))
```
</details>

### 💬 What does it mean?

- Which of Gemini's suggestions did you accept, and which did you reject? Why?
- How did you check that your improved code still calculates the same thing?
- Add this review to your AI log.

**✏️ MODEL ANSWER:**

*Answers will vary.* A strong answer accepts suggestions that make the code **clearer without changing the result** (meaningful names, comments, a rounded table, removing the intercept row) and explains any it **rejected**, for example a suggestion that adds complexity the task does not need, or one that would **change** the calculation.  
The check is to compare the numbers: the odds ratios (about **5.09** for distance and **18.33** for rain) and p-values must be identical before and after.  
Reviewing is useful, but **you** decide: AI suggestions are not automatically improvements.

---
## ✅ End of Code-along: Recap

You have learned to choose, fit and check the two main regression models for an EDA:

| Your outcome is… | Model | Read |
|---|---|---|
| A number | `smf.ols('y ~ x1 + x2 + C(group)', data=df).fit()` | Coefficients (holding others constant), p-values, adjusted R² |
| Yes/no (1/0) | `smf.logit('y ~ x1 + C(group)', data=df).fit(disp=0)` | Odds ratios `np.exp(model.params)`, p-values, predicted probabilities |

**Before fitting:** check candidate predictors for **overlap** (a correlation heatmap) and **leakage** (anything made from the outcome, or only known after it).

**After fitting:** a tidy table of results, interpreted in plain English.

---

### 💾 Before you leave

Make sure your notebook is saved to your Google Drive, if you have not done so already:
- **File → Save a copy in Drive**
- Name it: `Week10_CodeAlong_Regression_Models_YourName.ipynb`

---

### 🔮 Next: the Week 10 task

After the break you will build linear and logistic models for **FitLife**, use Gemini to review your code, and get a notebook ready for submission.